# Lecture 4.5 — Context Management: RunContextWrapper and Dependency Injection

**Section 04 — Running Agents, Results & Streaming**

In this notebook we go beyond the basics of context that you saw in Lectures 2.4 and 3.5. Here we focus on what happens **after** the run completes, how to read the full `Usage` breakdown, how context mutations flow between tools in the same run, how to carry context across multiple turns, and the metadata available on `ToolContext` for production patterns like audit logging.

## Cell 1 — Setting Up: Install the SDK

This notebook uses the OpenAI Agents SDK. We pin a specific version below so that the examples in this notebook behave consistently. If the package is already installed in your current Colab session (for example, from an earlier notebook you ran today), this cell will simply confirm it's present and move on quickly.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 874.3/874.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.6 MB/s eta 0:00:00


## Cell 2 — API Key Setup (Google Colab Secrets)

We retrieve the OpenAI API key from Colab's built-in Secrets manager rather than typing it into the notebook.

**Steps to add your secret in Colab:**
1. Click the key icon (🔑) in the left sidebar.
2. Click **Add new secret**.
3. Name it `OPENAI_API_KEY`.
4. Paste your key as the value.
5. Toggle **Notebook access** on for this notebook.

**Local (non-Colab) users:** set the environment variable in your terminal instead, e.g. `export OPENAI_API_KEY="sk-..."`, and skip the `userdata.get()` call below.

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Declaring MODEL_NAME

Every Agent in this notebook references a single `MODEL_NAME` variable instead of hardcoding a model string. Changing this one line updates the model used across the entire notebook.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4 — Imports

| Import | Purpose |
|---|---|
| `dataclass`, `field` | Build our `AppContext` dependency-injection object |
| `Reasoning` | Configure reasoning effort on `ModelSettings` (imported from `openai.types.shared`, **not** from `agents`) |
| `Agent`, `Runner` | Core SDK classes to define and run agents |
| `ModelSettings` | Tune reasoning effort and verbosity per agent |
| `RunContextWrapper` | The type used to annotate tool functions that need context access |
| `Usage` | The dataclass that tracks token and request usage |
| `function_tool` | Decorator that turns a Python function into a tool |
| `ToolContext` | An extended context available inside tools, imported from `agents.tool_context`, **not** the top-level `agents` package |

Two import locations are easy to get wrong: `Reasoning` comes from `openai.types.shared`, and `ToolContext` comes from `agents.tool_context`. Keep both in mind as you write your own tools later.

In [4]:
from dataclasses import dataclass, field
from typing import Any

from openai.types.shared import Reasoning

from agents import (
    Agent,
    ModelSettings,
    RunContextWrapper,
    Runner,
    Usage,
    function_tool,
)
from agents.tool_context import ToolContext

## Cell 5 — The Mental Model: Context as Dependency Injection

Before writing any code, let's build the complete mental model for context in the Agents SDK.

- You create a plain Python object (any type works: a dataclass, a Pydantic model, even a dict) and pass it to `Runner.run(..., context=your_object)`.
- The SDK wraps it internally in a `RunContextWrapper[T]`.
- Every agent, tool, hook, and guardrail involved in that run shares the **same** `RunContextWrapper` instance.
- Inside a tool function, `wrapper.context` gives you back your original object.
- `wrapper.usage` holds the aggregated `Usage` for the run so far.
- `wrapper.tool_input` holds structured input when the current run is executing inside `Agent.as_tool()`.
- `wrapper.turn_input` holds the input items the agent received for the current turn.

Two things to hold onto as you work through this notebook:

- **Context is never sent to the LLM.** It is purely local Python state that your tools and hooks can read and mutate. The model has no idea it exists.
- **`RunContextWrapper` is not thread-safe.** Do not share a single instance across concurrent `Runner.run()` calls — each concurrent run needs its own context object.

This is the same idea as dependency injection in a web framework: instead of reaching for global variables inside your tool functions, you receive your dependencies (a database connection, a user session, feature flags) through the context parameter.

## Cell 6 — A Real-World Dependency Injection Pattern

Here we build an `AppContext` dataclass that carries the kind of dependencies a real application would need: a user identity, a database connection string, feature flags, and a running log of queries made during the run.

Two tools read from and write to this shared context:

| Tool | Reads from context | Writes to context |
|---|---|---|
| `query_user_orders` | `user_id`, `username`, `db_connection_string` | Appends to `queries_made` |
| `check_feature_flag` | `feature_flags` | — |

Notice that neither tool receives these values as arguments from the model. They reach into `ctx.context` instead. This is the dependency injection pattern: the model only ever supplies `limit` or `flag_name`, and your application supplies everything else through context.

In [5]:
@dataclass
class AppContext:
    """Application context carrying dependencies."""
    user_id: str
    username: str
    db_connection_string: str
    feature_flags: dict = field(default_factory=dict)
    queries_made: list = field(default_factory=list)


@function_tool
def query_user_orders(
    ctx: RunContextWrapper[AppContext],
    limit: int,
) -> str:
    """Fetches recent orders for the current user.

    Args:
        limit: Maximum number of orders to return.
    """
    app = ctx.context
    app.queries_made.append(f"orders:{app.user_id}:limit={limit}")
    return (
        f"DB({app.db_connection_string}): Found {limit} "
        f"orders for user {app.username}."
    )


@function_tool
def check_feature_flag(
    ctx: RunContextWrapper[AppContext],
    flag_name: str,
) -> str:
    """Checks whether a feature flag is enabled.

    Args:
        flag_name: The feature flag name to check.
    """
    app = ctx.context
    enabled = app.feature_flags.get(flag_name, False)
    return f"Feature '{flag_name}': {'enabled' if enabled else 'disabled'}"


agent = Agent(
    name="Order Assistant",
    instructions=(
        "You are an order management assistant. "
        "Use tools to help users with their orders."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[query_user_orders, check_feature_flag],
)

app_ctx = AppContext(
    user_id="u001",
    username="Priya",
    db_connection_string="postgres://prod-db:5432/orders",
    feature_flags={
        "advanced_analytics": True,
        "beta_ui": False,
    },
)

result = await Runner.run(
    agent,
    "Show me my last 3 orders and check if advanced_analytics is enabled.",
    context=app_ctx,
)

print("Final output:", result.final_output)
print("Queries made:", app_ctx.queries_made)

Final output: Last 3 orders: 3 found for Priya.

advanced_analytics: enabled
Queries made: ['orders:u001:limit=3']


## Cell 7 — Reading `result.context_wrapper` After the Run

Once a run finishes, `result.context_wrapper` gives you the same `RunContextWrapper` instance that every tool call shared during the run. This is where you go to inspect the final state of your context object and, as we'll see next, the usage totals for the run.

Run the cell below and confirm that `queries_made` reflects both tool calls from the previous cell, even though we're reading it through `result.context_wrapper.context` rather than `app_ctx` directly. They are the same object.

In [6]:
print("=== result.context_wrapper ===")
print("Type:", type(result.context_wrapper).__name__)
print("Context type:", type(result.context_wrapper.context).__name__)
print("User:", result.context_wrapper.context.username)
print("Queries this run:", result.context_wrapper.context.queries_made)

=== result.context_wrapper ===
Type: RunContextWrapper
Context type: AppContext
User: Priya
Queries this run: ['orders:u001:limit=3']


## Cell 8 — The Usage Object in Full Detail

`result.context_wrapper.usage` gives you a `Usage` object with everything the SDK tracked during the run. Note the access path: it's `result.context_wrapper.usage`, **not** `result.usage`.

| Field | Meaning |
|---|---|
| `requests` | Total number of LLM API calls made during the run |
| `input_tokens` / `output_tokens` / `total_tokens` | Aggregated token counts across all requests |
| `input_tokens_details.cached_tokens` | Prompt cache hits within the input tokens |
| `output_tokens_details.reasoning_tokens` | Reasoning tokens used (relevant for GPT-5 models) |
| `request_usage_entries` | A list with one entry per individual LLM call, each carrying its own token breakdown |

A tool-using run typically makes at least two model calls: one that decides to call the tool, and one that reads the tool's result and produces the final answer. That's why you'll usually see two entries in `request_usage_entries` below. Summing the entries gives you the same totals as the aggregated fields, but the per-entry breakdown is what you need for granular cost tracking or debugging which call was expensive.

In [7]:
usage = result.context_wrapper.usage

print("=== Usage via context_wrapper ===")
print(f"Requests: {usage.requests}")
print(f"Input tokens: {usage.input_tokens}")
print(f"Output tokens: {usage.output_tokens}")
print(f"Total tokens: {usage.total_tokens}")
print(f"Cached input tokens: {usage.input_tokens_details.cached_tokens}")
print(f"Reasoning tokens: {usage.output_tokens_details.reasoning_tokens}")

print(f"\nPer-request breakdown ({len(usage.request_usage_entries)} entries):")
for i, entry in enumerate(usage.request_usage_entries):
    print(
        f"  Request {i + 1}: "
        f"in={entry.input_tokens} "
        f"out={entry.output_tokens} "
        f"total={entry.total_tokens}"
    )

=== Usage via context_wrapper ===
Requests: 2
Input tokens: 384
Output tokens: 74
Total tokens: 458
Cached input tokens: 0
Reasoning tokens: 0

Per-request breakdown (2 entries):
  Request 1: in=141 out=53 total=194
  Request 2: in=243 out=21 total=264


## Cell 9 — Context Mutation Across Tool Calls

The next example makes the sharing explicit. Two tools, `first_tool` and `second_tool`, are both called in the same run, and both receive the **same** `RunContextWrapper` instance. `second_tool` will print `queries_made` and see the entry that `first_tool` appended, even though they're separate function calls with no direct connection to each other in your code.

This is the pattern to reach for whenever you need tools to accumulate state or hand off information to each other during a single run: put it in context, not in a global variable.

**One caution:** this sharing is exactly why `RunContextWrapper` is not thread-safe. If you ran two `Runner.run()` calls concurrently against the same context object, both would be mutating it at once with no locking.

In [8]:
@function_tool
def first_tool(ctx: RunContextWrapper[AppContext]) -> str:
    """First tool — records a step."""
    ctx.context.queries_made.append("step_1")
    return "First tool completed."


@function_tool
def second_tool(ctx: RunContextWrapper[AppContext]) -> str:
    """Second tool — reads mutations from first tool."""
    # This sees the mutation from first_tool
    print(f"[second_tool] queries so far: {ctx.context.queries_made}")
    ctx.context.queries_made.append("step_2")
    return "Second tool completed."


multi_tool_agent = Agent(
    name="Multi Tool Agent",
    instructions="Call first_tool and then second_tool in that order.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[first_tool, second_tool],
)

shared_ctx = AppContext(
    user_id="u001",
    username="Priya",
    db_connection_string="postgres://prod-db:5432/orders",
)

result = await Runner.run(
    multi_tool_agent,
    "Run both tools in order.",
    context=shared_ctx,
)

print("Final output:", result.final_output)
print("Mutations accumulated:", shared_ctx.queries_made)

[second_tool] queries so far: ['step_1']
Final output: Done.
Mutations accumulated: ['step_1', 'step_2']


## Cell 10 — Context Across Multi-Turn Conversations

Context is entirely yours to manage across separate `Runner.run()` calls. If you pass the same context object to a second call, that second run continues mutating it, and you'll see accumulated state from both turns.

There's an important distinction to notice in the output below: `app_ctx.queries_made` accumulates across **both** turns, because it's the same Python object being mutated both times. But `result2.context_wrapper.usage` reflects **only** the second `Runner.run()` call. Usage is not automatically summed across separate calls the way context mutations are. If you need running usage totals across turns without doing it manually, that's what the Sessions feature (covered in a later update) automates for you.

In [9]:
result1 = await Runner.run(
    agent,
    "Show me my last 2 orders.",
    context=app_ctx,
)
print("Turn 1:", result1.final_output[:80])
print("Queries after turn 1:", app_ctx.queries_made)

result2 = await Runner.run(
    agent,
    result1.to_input_list() + [
        {
            "role": "user",
            "content": "Now check the beta_ui flag.",
        }
    ],
    context=app_ctx,
)
print("\nTurn 2:", result2.final_output[:80])
print("Queries after turn 2:", app_ctx.queries_made)
print("Turn 2 tokens:", result2.context_wrapper.usage.total_tokens)

Turn 1: Your last 2 orders are ready.
Queries after turn 1: ['orders:u001:limit=3', 'orders:u001:limit=2']

Turn 2: beta_ui is disabled.
Queries after turn 2: ['orders:u001:limit=3', 'orders:u001:limit=2']
Turn 2 tokens: 487


## Cell 11 — `ToolContext`: Tool-Specific Metadata

`ToolContext` extends `RunContextWrapper` with metadata about the specific tool call in progress. Import it from `agents.tool_context`, not from the top-level `agents` package.

| Field | Meaning |
|---|---|
| `tool_name` | Name of the tool being invoked |
| `tool_call_id` | Stable within a single invocation, unique across separate invocations. Useful as an idempotency key |
| `tool_arguments` | The raw JSON arguments string sent to the tool |
| `tool_call` | The underlying `ResponseFunctionToolCall`, when available |

The example below builds a simple audit log: every call to `audited_query` records the tool name, table, and `tool_call_id` into `queries_made`, giving you a traceable record of exactly what ran and when.

In [10]:
@function_tool
def audited_query(
    ctx: ToolContext[AppContext],
    table: str,
) -> str:
    """Performs an audited database query.

    Args:
        table: The database table to query.
    """
    app = ctx.context
    print(f"[AUDIT] tool_name: {ctx.tool_name}")
    print(f"[AUDIT] tool_call_id: {ctx.tool_call_id}")
    print(f"[AUDIT] tool_arguments: {ctx.tool_arguments}")
    app.queries_made.append(f"{ctx.tool_name}:{table}:id={ctx.tool_call_id}")
    return f"Query on {table} completed for {app.username}."


audit_agent = Agent(
    name="Audit Agent",
    instructions="Use audited_query to query database tables.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[audited_query],
)

fresh_ctx = AppContext(
    user_id="u002",
    username="Rahul",
    db_connection_string="postgres://prod-db:5432/main",
)

result = await Runner.run(
    audit_agent,
    "Query the users table.",
    context=fresh_ctx,
)

print("Final:", result.final_output)
print("Audit log:", fresh_ctx.queries_made)

[AUDIT] tool_name: audited_query
[AUDIT] tool_call_id: call_EVZRWfbDORE6HuOjPv8bvyrv
[AUDIT] tool_arguments: {"table":"users"}
Final: Query completed for users.
Audit log: ['audited_query:users:id=call_EVZRWfbDORE6HuOjPv8bvyrv']


## Cell 12 — Dependency Injection Patterns in Production

The `AppContext` pattern generalizes well beyond this notebook's examples. Here are common categories of things teams put in context on real projects:

| Pattern | What to put in context | Why |
|---|---|---|
| Database access | Connection string or session | Tools query the database without reaching for globals |
| HTTP clients | An `aiohttp.ClientSession` or similar | Reuse connection pools across tool calls |
| User session | `user_id`, `username`, permissions, tier | Personalise tools and gate features per user |
| Feature flags | `dict[str, bool]` | Enable or disable behaviour per user or rollout |
| Audit log | A mutable `list[str]` | Accumulate a tool call history across the run |
| Configuration | API URLs, environment name | Avoid hardcoding environment-specific values in tools |
| Rate limit state | Counters, timestamps | Enforce per-user rate limits from inside tools |

## Cell 13 — What Context Is Not

It's just as useful to be precise about the boundaries of context:

- **Not sent to the LLM.** The model never sees your context object, in any form.
- **Not serialised or persisted by the SDK.** If you need to persist context, you manage that yourself.
- **Not thread-safe.** Never share a single `RunContextWrapper` instance across concurrent `Runner.run()` calls.
- **Not conversation history.** History lives in `result.to_input_list()` or in Sessions, covered in a later update.
- **Not agent configuration.** Agent configuration (instructions, tools, model) lives on the `Agent` object itself, not in context.

With that boundary in place, you now have the full picture: context is your dependency injection system for a run, `result.context_wrapper` is how you read it back afterward, and `Usage` is the detailed cost ledger that comes along with it.